In [28]:
import pandas as pd
import bpe
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Metaspace

In [29]:
splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df_train = pd.read_parquet("hf://datasets/mteb/imdb/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/mteb/imdb/" + splits["test"])

In [30]:
print(df_test.head())

                                                text  label label_text
0  I love sci-fi and am willing to put up with a ...      0   negative
1  Worth the entertainment value of a rental, esp...      0   negative
2  its a totally average film with a few semi-alr...      0   negative
3  STAR RATING: ***** Saturday Night **** Friday ...      0   negative
4  First off let me say, If you haven't enjoyed a...      0   negative


In [31]:
print(df_train.head())

                                                text  label label_text
0  I rented I AM CURIOUS-YELLOW from my video sto...      0   negative
1  "I Am Curious: Yellow" is a risible and preten...      0   negative
2  If only to avoid making this type of film in t...      0   negative
3  This film was probably inspired by Godard's Ma...      0   negative
4  Oh, brother...after hearing about this ridicul...      0   negative


In [32]:
print("Train label distribution:")
print(df_train['label'].value_counts())
print("Total amount of train entries:")
print(len(df_train))
print("Test label distribution:")
print(df_test['label'].value_counts())
print("Total amount of test entries:")
print(len(df_test))

Train label distribution:
label
1    12472
0    12432
Name: count, dtype: int64
Total amount of train entries:
24904
Test label distribution:
label
1    12412
0    12266
Name: count, dtype: int64
Total amount of test entries:
24678


The dataset is divided into a train set of 24904 rows and a test set of 24678 rows. Each row contains four columns: the review text, a label (0 for negative, 1 for positive), a label_text (negative or positive) and split (train or test). Both sets are almost equally balanced: the training set consists of 12472 positive (50.08%) and 12432 negative (50.02%) reviews, while the test set contains 12412 positive (50.30%) and 12266 negative (49.70%) reviews. Because the classes are evenly distributed, accuracy can be used as an evaluation metric for this analysis.

In [34]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Metaspace()
trainer = BpeTrainer(vocab_size=1000, min_frequency=2, special_tokens=["[UNK]"])
corpus = df_train['text'].tolist()
tokenizer.train_from_iterator(corpus, trainer)